In [ ]:
import sys
import os
from pathlib import Path

ROOT = Path().resolve().parent.parent
sys.path.insert(0, str(ROOT))

print("Añadido al path:", ROOT)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10,6)

pd.set_option("display.max_columns", None)

In [ ]:
import io
import os
import matplotlib.pyplot as plt

from utils.funciones_minio import crear_cliente_minio, bajar_minio
from utils.config import PATH_PRIMARIOS_LIMPIO

OBJ_VIVIENDAS_VENTA = "viviendas_venta.parquet"
OBJ_VIVIENDAS_ALQUILER = "viviendas_alquiler.parquet"


In [ ]:
client = crear_cliente_minio()

In [ ]:
df_venta = bajar_minio(client, PATH_PRIMARIOS_LIMPIO, OBJ_VIVIENDAS_VENTA)
if not isinstance(df_venta, pd.DataFrame):
        df_venta = pd.read_parquet(io.BytesIO(df_venta))
df_alquiler = bajar_minio(client, PATH_PRIMARIOS_LIMPIO, OBJ_VIVIENDAS_ALQUILER)
if not isinstance(df_alquiler, pd.DataFrame):
        df_alquiler = pd.read_parquet(io.BytesIO(df_alquiler))

En este archivo haremos un análisis estadístico sobre el campo Descripción de cada anuncio, así como su relación con los el tipo de Anunciante

In [ ]:
df_venta["tipo"] = "venta"
df_alquiler["tipo"] = "alquiler"

df = pd.concat([df_venta, df_alquiler])

Podemos agrupar Agente Pro y Profesional en un solo tipo debido a sus similitudes: Intermediario

In [ ]:
def agrupar_tipo(x):
    if x == "Particular":
        return "Particular"
    elif x in ["Agente Pro", "Profesional"]:
        return "Intermediario"
    elif x == "Promotora":
        return "Promotora"

df["grupo"] = df["Anuncia"].apply(agrupar_tipo)

df["grupo"].value_counts()

In [ ]:
df["longitud"] = df["Descripcion"].str.len()
df["num_palabras"] = df["Descripcion"].str.split().str.len()

df.groupby("grupo")[["longitud","num_palabras"]].mean().sort_values("longitud")

Vemos que los anuncios de las promotoras e intermediarios tienden a ser más largos que los de los particulares

In [ ]:
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer

nltk.download("stopwords")

stopwords_es = stopwords.words("spanish")

vectorizer = CountVectorizer(
    stop_words=stopwords_es,
    max_features=40
)
df = df.reset_index(drop=True)
X = vectorizer.fit_transform(df["Descripcion"].fillna(""))

In [ ]:
palabras = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out())
palabras["grupo"] = df["grupo"]

frecuencias = palabras.groupby("grupo").mean().T
frecuencias.sort_values("Promotora", ascending=False).head(15)

In [ ]:
frecuencias.sort_values("Particular", ascending=False).head(15)

In [ ]:
frecuencias.sort_values("Intermediario", ascending=False).head(15)

Vectorizamos el texto y comparamos las frecuencias de las palabras. Vemos que algunos terminos claramente se usan más por parte de promotoras e intermediarios, como son "ofrece", "zonas", etc.

In [ ]:
df["keyword_empresa"] = df["Descripcion"].str.contains(
    "inmobiliaria|honorarios|gestión|asesor|obra nueva|promoción",
    case=False, regex=True
)

pd.crosstab(df["keyword_empresa"], df["grupo"], normalize="index")

Aquí vemos determinadas palabras clave que diferencian las intermediarias de los demás anunciantes

In [ ]:
df["ratio_mayus"] = df["Descripcion"].apply(
    lambda x: sum(1 for c in x if c.isupper()) / len(x) if isinstance(x,str) else 0
)

df.groupby("grupo")["ratio_mayus"].mean()

In [ ]:
df["num_exclamaciones"] = df["Descripcion"].str.count("!")

df.groupby("grupo")["num_exclamaciones"].mean()

Si bien el análisis de mayúsculas no muestran una diferenciación clara, los intermediarios si que usan muchos más signos de exclamación que las otras.

Tras este análisis podemos observar claramente que existen diferencias significativas entre los tipos diferentes de anunciantes. Por tanto, consideramos que puede ser interesante implementar un modelo que clasifique un anuncio entre estas clases solo conociendo la descripción.